In [1]:
%reset -f
import math
import os
import warnings
from collections.abc import Callable
from math import pow, sqrt

import forallpeople as units
import handcalcs
import handcalcs.render
import numpy as np

# import polars as pl
import pandas as pd
import sympy as sp
from structural_tools.notebook import (
    check_value,  # noqa: F401
    display_table,
    display_text,
    set_params_columns,
    sig_figs,  # noqa: F401
)
from structural_tools.seismic.asce.parameters import LOAD_FACTOR_SEISMIC_ASD, RiskCategory, SeismicParameters, SiteClass

units.environment(env_name="structural", top_level=True)
handcalcs.set_option("custom_symbols", {"star": "^*", "__": ",", "plus": "+", "minus": "-"})
handcalcs.set_option("greek_exclusions", ["psi"])
handcalcs.set_option("display_precision", 4)
handcalcs.set_option("param_columns", 1)
handcalcs.set_option("latex_block_start", "\\begin{equation*}")
handcalcs.set_option("latex_block_end", "\\end{equation*}")

# Effective Seismic Weight Calculation

We calculate the effective seismic weight with unit weights and square footages.

## Unit Weights

Weights will be assigned with area unit weights.

In [2]:
set_params_columns(4)

In [3]:
%%render params
U_corridor = 35 / 1000 * ksf
U_floor = 35 / 1000 * ksf
U_roof = 25 / 1000 * ksf
U_wall = 12 / 1000 * ksf

<IPython.core.display.Latex object>

## Building Geometry

The lengths and widths of the weight areas are consistent across levels 2, 3, and 4.

In [4]:
set_params_columns(5)

In [5]:
%%render params
L_floor = 230 * ft
w_floor = (30 + 9 / 12) * ft
L_corridor = 230 * ft
w_corridor = 5.5 * ft
w_building = (w_floor * 2) + w_corridor

<IPython.core.display.Latex object>

The wall heights vary across the levels.

In [6]:
%%render params
h_wall__1 = 11.5 * ft
h_wall__2 = 10.5 * ft
h_wall__3 = 10.5 * ft
h_wall__4 = 10 * ft

<IPython.core.display.Latex object>

There is a small jog in the building floorplan. This does not affect the area, but the perimeter will be longer.

In [7]:
%%render params
L_jog = 5 * ft

<IPython.core.display.Latex object>

In [8]:
%%render
P_wall = (2 * L_floor) + (2 * w_building) + (2 * L_jog)

<IPython.core.display.Latex object>

Areas can now be calculated. Note that the areas of the walls are the actual wall areas; they have not been distributed to each floor based on tributary area, yet.

In [9]:
%%render 
A_floor = 2 * (L_floor * w_floor)
# This is a slight overestimate as the corridor doesn't stretch the full length of the building
A_corridor = L_corridor * w_corridor
A_level = L_floor * w_building
A_wall__1 = h_wall__1 * P_wall
A_wall__2 = h_wall__2 * P_wall
A_wall__3 = h_wall__3 * P_wall
A_wall__4 = h_wall__4 * P_wall

<IPython.core.display.Latex object>

## Component Weights

The typical floor and corridor weights are

In [10]:
%%render
W_floor__typ = U_floor * A_floor
W_corridor__typ = U_corridor * A_corridor

<IPython.core.display.Latex object>

This applies to levels 2, 3, and 4.

In [11]:
set_params_columns(3)

In [12]:
%%render params
W_floor__2 = W_floor__typ
W_floor__3 = W_floor__typ
W_floor__4 = W_floor__typ
W_corridor__2 = W_corridor__typ
W_corridor__3 = W_corridor__typ
W_corridor__4 = W_corridor__typ

<IPython.core.display.Latex object>

The roof weight is

In [13]:
%%render
W_roof = U_roof * A_level

<IPython.core.display.Latex object>

The exterior walls are divided in half to the floor above and below.

In [14]:
%%render
W_wall__2 = U_wall * (A_wall__2 / 2 + A_wall__1 / 2)
W_wall__3 = U_wall * (A_wall__3 / 2 + A_wall__2 / 2)
W_wall__4 = U_wall * (A_wall__4 / 2 + A_wall__3 / 2)
W_wall__roof = U_wall * (A_wall__4 / 2)

<IPython.core.display.Latex object>

## Level Weights

Each level's weight is

In [15]:
%%render 
W_level__2 = W_floor__2 + W_corridor__2 + W_wall__2
W_level__3 = W_floor__3 + W_corridor__3 + W_wall__3
W_level__4 = W_floor__4 + W_corridor__4 + W_wall__4
# be sure to add the weight of the truss later
W_level__roof = W_roof + W_wall__roof

<IPython.core.display.Latex object>

## Total Weight

The total weight of the structure is

In [16]:
%%render long
W_total = W_level__2 + W_level__3 + W_level__4 + W_level__roof

<IPython.core.display.Latex object>

# Seismic Equivalent Lateral Force (ELF) Analysis

We will be using the equivalent lateral force method prescribed by ASCE 7-16 Section 12.8. We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

## Seismic Parameters

In [17]:
S_S = 0.887  # g
S_1 = 0.410  # g
seismic_params = SeismicParameters(s_1=S_1, s_s=S_S, risk_category=RiskCategory.II, site_class=SiteClass.D)

Seismic design parameters are obtained from the geotech report and ASCE 7-16 Sec. 11 and 12.

In [18]:
set_params_columns(5)

In [19]:
%%render params
R = 6.5
Omega_0 = 3.0
C_d = 4.0
rho = 1.0

<IPython.core.display.Latex object>

In [20]:
%%render params
I_e = seismic_params.i_e
S_S = seismic_params.s_s  # g
S_1 = seismic_params.s_1  # g
F_a = seismic_params.f_a
F_v = seismic_params.f_v
S_MS = seismic_params.s_ms  # g
S_M1 = seismic_params.s_m1  # g
S_DS = seismic_params.s_ds  # g
S_D1 = seismic_params.s_d1  # g
SDC = seismic_params.sdc
C_u = 1.4
T_L = 16  # s

<IPython.core.display.Latex object>

## Base Shear

We must calculate the period

In [21]:
set_params_columns(1)

In [22]:
%%render params
h_top__roof = (58 + 3 / 12) * ft
h_bottom__roof = 42.5 * ft
h_n = np.mean([h_top__roof, h_bottom__roof])  # ASCE 7-16 Sec. 11.2

<IPython.core.display.Latex object>

In [23]:
%%render
h_n__check = check_value(h_n, 65 * ft, "leq")

<IPython.core.display.Latex object>

We will approximate the building period.

In [24]:
h_n = float(h_n)  # to prevent weirdness with units

In [25]:
%%render
C_t = 0.02  # ASCE 7-16 Table 12.8-2
x = 0.75  # ASCE 7-16 Table 12.8-2
T_a = C_t * h_n**x  # ASCE 7-16 Eq. 12.8-7

<IPython.core.display.Latex object>

Because we are not using an automated analysis software, we can make the following assumption:

In [26]:
%%render
T = T_a  # ASCE 7-16 Sec. 12.8.2

<IPython.core.display.Latex object>

With the period, we can calculate the seismic response coefficient.

In [27]:
%%render
C_s = S_DS / (R / I_e)  # ASCE 7-16 Eq. 12.8-2

<IPython.core.display.Latex object>

We must check that $C_s$ does not exceed:

In [28]:
%%render long
# fmt: off
if T <= T_L:  C_s__check__max = S_D1 / (T * (R / I_e));              C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
elif T > T_L: C_s__check__max = (S_D1 * T_L) / (T**2 * (R / I_e));   C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
# fmt: on

<IPython.core.display.Latex object>

And we must also check that $C_s$ is not less than:

In [29]:
%%render
C_s__check__min__1 = 0.044 * S_DS * I_e  # ASCE 7-16 Eq. 12.8-5
C_s__check__min = check_value(C_s, C_s__check__min__1, "geq")

<IPython.core.display.Latex object>

as well as:

In [30]:
%%render
C_s__check__min = check_value(C_s, 0.01, "geq")  # ASCE 7-16 Eq. 12.8-5

<IPython.core.display.Latex object>

and finally:

In [31]:
%%render
# fmt: off
if S_1 >= 0.6:  C_s__check__min__3 = 0.5 * S_1 / (R / I_e); C_s__check__min = check_value(C_s, C_s__check__min__3, "geq")  # ASCE 7-16 Eq. 12.8-6
elif S_1 < 0.6: C_s__check__min = "\\text{skipping check}"  # ASCE 7-16 Eq. 12.8-6
# fmt: on

<IPython.core.display.Latex object>

And now, we can calculate the base shear, $V$

In [32]:
%%render
V = C_s * W_total  # ASCE 7-16 Eq. 12.8-1

<IPython.core.display.Latex object>

## Distribute the seismic forces

With the base shear calculated, we must now distribute the base shear on each level.

First, we must calculate the structural period exponent, $k$.

In [33]:
%%render
# fmt: off
if T <= 0.5: k = 1
elif T >= 2.5: k = 2
elif T > 0.5 and T < 2.5: k = 1 + ((T - 0.5) * (2 - 1)) / (2.5 - 0.5)
# fmt: on

<IPython.core.display.Latex object>

Now, we can distribute the forces per floor,

$$
F_x = C_{vx} V\;\;\text{(ASCE 7-16 Eq. 12.8-11)}
$$

As well as cumulatively from roof to ground.

$$
V_x = \sum_{i=x}^n F_i\;\; \text{(ASCE 7-16 Eq. 12.8-11)}
$$

In [34]:
# Load in basic data
seismic_loads = pd.DataFrame(
    {
        "Level": ["Ground", "Level 2", "Level 3", "Level 4", "Roof"],
        "floor height": [
            float(h_wall__1),
            float(h_wall__2),
            float(h_wall__3),
            float(h_wall__4),
            0,
        ],
        "floor weight": [
            0,
            float(W_level__2),
            float(W_level__3),
            float(W_level__4),
            float(W_level__roof),
        ],
    },
)
seismic_loads = seismic_loads.set_index("Level")

# Calc cumulative elevation
elevations = [0]
for height in seismic_loads["floor height"][:-1]:
    elevations.append(height + elevations[-1])
seismic_loads.insert(2, "elevation", elevations)

# Calc coefficients and story forces ASCE 7-16 Eq. 12.8-11 and 12.8-12
seismic_loads["wxhxk"] = seismic_loads["floor weight"] * seismic_loads["elevation"] ** float(k)
cumulative_wxhxk = seismic_loads["wxhxk"].sum()
seismic_loads["C_vx"] = seismic_loads["wxhxk"] / cumulative_wxhxk
seismic_loads["F_x"] = seismic_loads["C_vx"] * float(V)

# Calc story shear forces ASCE 7-16 Eq. 12.8-13
story_shears = [0]
for shear in reversed(list(seismic_loads["F_x"][1:])):
    story_shears.insert(0, shear + story_shears[0])
seismic_loads["V_x"] = story_shears

# Calc overturning moments
floor_moments = seismic_loads["V_x"] * seismic_loads["floor height"]
overturning_moments = [0]
for floor_moment in reversed(list(floor_moments[:-1])):
    overturning_moments.insert(0, overturning_moments[0] + floor_moment)
seismic_loads["OTM"] = overturning_moments

In [35]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "floor height": "$h_{floor}$ [ft]",
    "floor weight": "$w_x$ [kip]",
    "elevation": "$h_x$ [ft]",
    "wxhxk": "$w_x h_x^k$",
    "C_vx": "$C_{vx}$",
    "F_x": "$F_x$ [kip]",
    "V_x": "$V_x$ [kip]",
    "OTM": "OTM [kip-ft]",
}

Calculating for each floor leads to the following \autoref{tab:seismic_forces_calculation_table}

In [36]:
# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
display_table(
    dataframe=seismic_loads.iloc[::-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Seismic Forces Calculation Table",
    label="tab:seismic_forces_calculation_table",
)

,$h_{floor}$ [ft],$w_x$ [kip],$h_x$ [ft],$w_x h_x^k$,$C_{vx}$,$F_x$ [kip],$V_x$ [kip],OTM [kip-ft]
Level,,,,,,,,
Roof,0.0,421.490,42.5,17913.325,0.306127,72.387392,0.000000,0.000000
Level 4,10.0,613.642,32.5,19943.365,0.340819,80.590744,72.387392,723.873920
Level 3,10.5,615.454,22.0,13539.988,0.231389,54.714824,152.978136,2330.144343
Level 2,10.5,619.078,11.5,7119.397,0.121666,28.769342,207.692959,4510.920416
Ground,11.5,0.000,0.0,0.000,0.000000,0.000000,236.462302,7230.236885


# Diaphragm and Shear Wall Design

## Redundancy Factor

The redundancy factor, $\rho$, is either 1.3 or 1.0 according to ASCE 7-16 12.3.4.

No single story resists more than 35% of the shear force (see $C_{vx}$ in \autoref{tab:seismic_forces_calculation_table}), so all values of $\rho$ are $1.0$.

## Shear Wall Design

We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

In [37]:
# Set up diaphragm dataframe
diaphragm_rigidity = ["Flexible", "Rigid"]
levels = ["Roof", "Level 4", "Level 3", "Level 2"]
walls = {
    "B-2": "N-S",
    "B-2.4": "N-S",
    "B-3.7": "N-S",
    "B-4": "N-S",
    "B-5": "N-S",
    "B-5.1": "N-S",
    "B-6": "N-S",
    "B-6.1": "N-S",
    "B-6.8": "N-S",
    "B-7": "N-S",
    "B-7.8": "N-S",
    "B-8": "N-S",
    "B-8.2": "N-S",
    "B-8.9": "N-S",
    "B-9": "N-S",
    # E-W
    "B-B--B-1": "E-W",
    "B-B--B-2.5": "E-W",
    "B-B--B-3.5": "E-W",
    "B-B--B-4": "E-W",
    "B-B--B-4.8": "E-W",
    "B-B--B-5.5": "E-W",
    "B-B--B-6.5": "E-W",
    "B-B--B-7.9": "E-W",
    "B-B--B-8.8": "E-W",
    "B-C--B-1.8": "E-W",
    "B-C--B-3.2": "E-W",
    "B-C--B-3.8": "E-W",
    "B-C--B-4.6": "E-W",
    "B-C--B-5.5": "E-W",
    "B-C--B-6.5": "E-W",
    "B-C--B-6.9": "E-W",
    "B-C--B-8.5": "E-W",
    "B-C--B-9.2": "E-W",
}

index = pd.MultiIndex.from_product(
    [levels, walls.keys()],
    names=["Level", "Wall"],
)

shear_walls = pd.DataFrame(index=index)
shear_walls["Direction"] = shear_walls.index.get_level_values("Wall").map(walls)
flexible = shear_walls.copy()
flexible_ns = flexible[flexible["Direction"] == "N-S"].copy()
flexible_ew = flexible[flexible["Direction"] == "E-W"].copy()
rigid = shear_walls.copy()
rigid_ns = rigid[rigid["Direction"] == "N-S"].copy()
rigid_ew = rigid[rigid["Direction"] == "E-W"].copy()

### Flexible Diaphragm Assumption

We will uniformly distribute the shear forces across the diaphragms based on tributary areas of each level. This removes some nuance in the distribution of floor weight distribution (corridors are 17% heavier than the rest of the floors, for instance), but the vast majority of the area of each level is typical floor weight (30 psf) instead of corridor weight (35 psf).

$$
f_{p,x} = F_x / A_{level}
$$

In [38]:
%%render
A_level
f_p__roof = seismic_loads.loc["Roof", "F_x"] * 1000 * lb / A_level
f_p__4 = seismic_loads.loc["Level 4", "F_x"] * 1000 * lb / A_level
f_p__3 = seismic_loads.loc["Level 3", "F_x"] * 1000 * lb / A_level
f_p__2 = seismic_loads.loc["Level 2", "F_x"] * 1000 * lb / A_level

<IPython.core.display.Latex object>

#### North-South Direction

Most of the shear walls are not in line which presents an issue: if we assume a diaphragm is the full depth, there will be an extremely long collector element without a shear wall on both sides. We will avoid this by just assuming the flexible diaphragm does not extend the full depth and instead is just tributary to the nearest wall (\autoref{fig:diaphragm_trib_area}). This means that there are essentially two diaphragms through the depth of the building. This is a rough approximation to ensure all the shear load is resolved in the shear walls. \

The walls that are close to inline (B-5 and B-5.1, B-6 and B-6.1, and B-9 and B-9.1) in reality will be treated as inline and have collectors to transfer the shear load from the diaphragm to those walls.

Proper detailing will be done to ensure continuity through the depth of the diaphragm.

![Shear walls (thick red lines) in plan view with their (colored) tributary areas.](images/shear_wall_tributary_area.png){width=60% #fig:diaphragm_trib_area}

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ns} through \autoref{tab:shear_wall_forces_level2_ns}.

Shear wall sheathing will be assumed to have the following properties:

1. WSP Sheathing
2. 15/32" thickness
3. 10d common nails
4. Plywood panels
5. Framing uses Douglas-Fir-Larch (no specific gravity adjustment)

Shear capacity and stiffness values come from **AWC SDPWS 2021 Table 4.3A**.

Notes: 

- Cumulative shear force, $F_{x,cum}$, is cumulative from the Roof downwards. For example, Level 3 includes the shear forces from the Roof, Level 4, and Level 3.

In [39]:
from structural_tools.seismic.timber.sheathing import (
    DesignMethod,
    LoadCase,
    Nail,
    PanelType,
    Sheathing,
    SheathingMaterial,
    get_sheathing_properties,
)

# Set baseline sheathing properties. We set the number of sheathed sides later
sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
)

In [40]:
# Plan dimensions
l_B1__B2 = (31 + 11 / 12) * ft
l_B2__B3 = (17 + 11 / 12) * ft
l_B3__B4 = (26 + 4 / 12) * ft
l_B4__B5 = (21 + 4 / 12) * ft
l_B5__B6 = (24 + 8 / 12) * ft
l_B6__B7 = (34 + 4 / 12) * ft
l_B7__B8 = (26 + 1 / 12) * ft
l_B8__B9 = (21 + 4 / 12) * ft
l_B9__B10 = (31 + 1 / 12) * ft

l_B1__B24 = (40 + 1 / 12) * ft
l_B24__B35 = (23 + 8 / 12) * ft
l_B35__B51 = (34 + 4 / 12) * ft
l_B51__B61 = (24 + 8 / 12) * ft
l_B61__B68 = (24 + 8 / 12) * ft
l_B68__B82 = (38 + 3 / 12) * ft
l_B82__B89 = (17 + 11 / 12) * ft
l_B89__B10 = (31 + 5 / 12) * ft

l_B37__B4 = (8 + 10 / 12) * ft
l_B3__B37 = l_B3__B4 - l_B37__B4
l_B78__B82 = (8 + 10 / 12) * ft
l_B68__B78 = l_B68__B82 - l_B78__B82

d_trib_ns = 67 / 2 * ft

In [41]:
# Wall tributary widths and lengths
t_w__B2 = l_B1__B2 + (l_B2__B3 + l_B3__B37) / 2
t_w__B24 = (l_B1__B24 - L_jog) + ((l_B24__B35 + l_B35__B51) / 2)
t_w__B37 = ((l_B2__B3 + l_B3__B37) / 2) + (l_B37__B4 / 2)
t_w__B4 = (l_B37__B4 / 2) + (l_B4__B5 / 2)
t_w__B5 = (l_B4__B5 / 2) + (l_B5__B6 / 2)
t_w__B51 = ((l_B24__B35 + l_B35__B51) / 2) + (l_B51__B61 / 2)
t_w__B6 = (l_B5__B6 / 2) + (l_B6__B7 / 2)
t_w__B61 = (l_B51__B61 / 2) + (l_B61__B68 / 2)
t_w__B68 = (l_B61__B68 / 2) + (l_B68__B78 / 2)
t_w__B7 = (l_B6__B7 / 2) + (l_B7__B8 / 2)
t_w__B78 = (l_B68__B78 / 2) + (l_B78__B82 / 2)
t_w__B8 = (l_B7__B8 / 2) + (l_B8__B9 / 2)
t_w__B82 = (l_B78__B82 / 2) + (l_B82__B89 / 2)
t_w__B89 = (l_B82__B89 / 2) + l_B89__B10
t_w__B9 = (l_B8__B9 / 2) + (l_B9__B10 - L_jog)

t_l__B2 = float(d_trib_ns)
t_l__B24 = float(d_trib_ns)
t_l__B37 = float(d_trib_ns)
t_l__B4 = float(d_trib_ns)
t_l__B5 = float(d_trib_ns)
t_l__B51 = float(d_trib_ns)
t_l__B6 = float(d_trib_ns)
t_l__B61 = float(d_trib_ns)
t_l__B68 = float(d_trib_ns)
t_l__B7 = float(d_trib_ns)
t_l__B78 = float(d_trib_ns)
t_l__B8 = float(d_trib_ns)
t_l__B82 = float(d_trib_ns)
t_l__B89 = float(d_trib_ns)
t_l__B9 = float(d_trib_ns)

In [42]:
flexible_ns.loc["Roof", "tributary weight"] = float(f_p__roof)
flexible_ns.loc["Level 4", "tributary weight"] = float(f_p__4)
flexible_ns.loc["Level 3", "tributary weight"] = float(f_p__3)
flexible_ns.loc["Level 2", "tributary weight"] = float(f_p__2)

# Shear wall height is for the floor below
flexible_ns.loc["Roof", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ns.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ns.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ns.loc["Level 2", "wall height"] = float(seismic_loads.loc["Ground", "floor height"])

# Add wall tributary widths and wall lengths to df
l_ba__bb = 29 + 9 / 12
l_bc__bd = 29 + 9 / 12
l_stairwall = 20
wall_tributary = {
    "B-2": {"trib_width": float(t_w__B2), "trib_depth": t_l__B2, "wall_length": l_ba__bb},
    "B-2.4": {"trib_width": float(t_w__B24), "trib_depth": t_l__B24, "wall_length": l_bc__bd},
    "B-3.7": {"trib_width": float(t_w__B37), "trib_depth": t_l__B37, "wall_length": l_stairwall},
    "B-4": {"trib_width": float(t_w__B4), "trib_depth": t_l__B4, "wall_length": l_stairwall},
    "B-5": {"trib_width": float(t_w__B5), "trib_depth": t_l__B5, "wall_length": l_ba__bb},
    "B-5.1": {"trib_width": float(t_w__B51), "trib_depth": t_l__B51, "wall_length": l_bc__bd},
    "B-6": {"trib_width": float(t_w__B6), "trib_depth": t_l__B6, "wall_length": l_ba__bb},
    "B-6.1": {"trib_width": float(t_w__B61), "trib_depth": t_l__B61, "wall_length": l_bc__bd},
    "B-6.8": {"trib_width": float(t_w__B68), "trib_depth": t_l__B68, "wall_length": l_bc__bd},
    "B-7": {"trib_width": float(t_w__B7), "trib_depth": t_l__B7, "wall_length": l_ba__bb},
    "B-7.8": {"trib_width": float(t_w__B78), "trib_depth": t_l__B78, "wall_length": l_stairwall},
    "B-8": {"trib_width": float(t_w__B8), "trib_depth": t_l__B8, "wall_length": l_ba__bb},
    "B-8.2": {"trib_width": float(t_w__B82), "trib_depth": t_l__B82, "wall_length": l_stairwall},
    "B-8.9": {"trib_width": float(t_w__B89), "trib_depth": t_l__B89, "wall_length": l_bc__bd},
    "B-9": {"trib_width": float(t_w__B9), "trib_depth": t_l__B9, "wall_length": l_ba__bb},
}

flexible_ns["wall length"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary width"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary depth"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary area"] = flexible_ns["tributary depth"] * flexible_ns["tributary width"]

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ns["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up. Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ns["line"] = flexible_ns.index.get_level_values("Wall").str.split("--").str[0]
# Get tributary area and wall length for the shear line
flexible_ns["line tributary area"] = flexible_ns.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ns["line wall length"] = flexible_ns.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ns["line shear demand"] = flexible_ns["line tributary area"] * flexible_ns["tributary weight"]
flexible_ns["line unit shear"] = flexible_ns["line shear demand"] / flexible_ns["line wall length"]
flexible_ns["line cumulative shear demand"] = flexible_ns.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ns["line cumulative unit shear"] = (
    flexible_ns["line cumulative shear demand"] / flexible_ns["line wall length"]
)

flexible_ns["shear demand"] = flexible_ns["line unit shear"] * flexible_ns["wall length"]
flexible_ns["cumulative shear demand"] = flexible_ns.groupby(level="Wall")["shear demand"].cumsum()
flexible_ns["unit shear demand"] = flexible_ns["cumulative shear demand"] / flexible_ns["wall length"]
flexible_ns["adjusted unit shear demand"] = flexible_ns["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ns["sheathed sides"] = 1
flexible_ns["aspect ratio"] = flexible_ns["wall height"] / flexible_ns["wall length"]

#### Check aspect ratios

We must check the aspect ratios of the shear walls to ensure we don't need to apply the aspect ratio factor in SDPWS Sec. 4.3.3.2. The maximum aspect ratio of all N-S walls is:

In [43]:
%%render
ratio_max = check_value(flexible_ns["aspect ratio"].max(), 2, "<")

<IPython.core.display.Latex object>

In [44]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [45]:
flexible_ns[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(flexible_ns[["adjusted unit shear demand", "sheathed sides"]], sheathing=sheathing)
)

shear_dcr_check_value = 1

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ns["shear dcr"] > shear_dcr_check_value).any():
    flexible_ns.loc[flexible_ns["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ns.loc[
        flexible_ns["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ns.loc[
            flexible_ns["shear dcr"] > shear_dcr_check_value, ["adjusted unit shear demand", "sheathed sides"]
        ],
        sheathing=sheathing,
    )

In [46]:
# Update dataframe
flexible = flexible.reindex(columns=flexible_ns.columns)
flexible["line"] = flexible["line"].astype("string")
flexible.update(flexible_ns)

In [47]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$w_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [lbf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [48]:
def highlight_insufficient_capacity(row):
    styles = [""] * len(row)

    if row["$0.7 v$ [plf]"] > row["$v_{cap}$ [plf]"]:
        demand_idx = row.index.get_loc("$0.7 v$ [plf]")
        capacity_idx = row.index.get_loc("$v_{cap}$ [plf]")

        styles[demand_idx] = "background-color: red; font-weight: bold"
        styles[capacity_idx] = "background-color: red; font-weight: bold"

    return styles

In [49]:
display_table(
    dataframe=flexible_ns,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Roof",
    label="tab:shear_wall_forces_roof_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 4",
    label="tab:shear_wall_forces_level4_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 3",
    label="tab:shear_wall_forces_level3_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 2",
    label="tab:shear_wall_forces_level2_ns",
)

,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
B-2,4.69743,1662.437500,7809.183325,7809.183325,29.75,262.493557,183.745490,310.714286,1,6.0
B-2.4,4.69743,2146.791667,10084.402984,10084.402984,29.75,338.971529,237.280070,310.714286,1,6.0
B-3.7,4.69743,741.187500,3481.676193,3481.676193,20.00,174.083810,121.858667,310.714286,1,6.0
B-4,4.69743,505.291667,2373.572094,2373.572094,20.00,118.678605,83.075023,310.714286,1,6.0
B-5,4.69743,770.500000,3619.369601,3619.369601,29.75,121.659482,85.161638,310.714286,1,6.0
B-5.1,4.69743,1384.666667,6504.374356,6504.374356,29.75,218.634432,153.044102,310.714286,1,6.0
B-6,4.69743,988.250000,4642.234923,4642.234923,29.75,156.041510,109.229057,310.714286,1,6.0
B-6.1,4.69743,826.333333,3881.642761,3881.642761,29.75,130.475387,91.332771,310.714286,1,6.0
B-6.8,4.69743,905.895833,4255.382013,4255.382013,29.75,143.038051,100.126636,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
B-2,5.229769,1662.437500,8694.164450,16503.347775,29.75,554.734379,388.314065,460.714286,1,4.0
B-2.4,5.229769,2146.791667,11227.224957,21311.627941,29.75,716.357242,501.450069,600.000000,1,3.0
B-3.7,5.229769,741.187500,3876.239566,7357.915758,20.00,367.895788,257.527052,310.714286,1,6.0
B-4,5.229769,505.291667,2642.558800,5016.130894,20.00,250.806545,175.564581,310.714286,1,6.0
B-5,5.229769,770.500000,4029.537176,7648.906777,29.75,257.106110,179.974277,310.714286,1,6.0
B-5.1,5.229769,1384.666667,7241.487099,13745.861455,29.75,462.045763,323.432034,460.714286,1,4.0
B-6,5.229769,988.250000,5168.319421,9810.554345,29.75,329.766533,230.836573,310.714286,1,6.0
B-6.1,5.229769,826.333333,4321.532623,8203.175384,29.75,275.736988,193.015891,310.714286,1,6.0
B-6.8,5.229769,905.895833,4737.626136,8993.008149,29.75,302.285988,211.600192,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
B-2,3.550605,1662.437500,5902.658971,22406.006745,29.75,753.143084,527.200159,600.000000,1,3.0
B-2.4,3.550605,2146.791667,7622.409317,28934.037258,29.75,972.572681,680.800877,769.642857,1,2.0
B-3.7,3.550605,741.187500,2631.664075,9989.579834,20.00,499.478992,349.635294,460.714286,1,4.0
B-4,3.550605,505.291667,1794.091140,6810.222033,20.00,340.511102,238.357771,310.714286,1,6.0
B-5,3.550605,770.500000,2735.741185,10384.647963,29.75,349.063797,244.344658,310.714286,1,6.0
B-5.1,3.550605,1384.666667,4916.404449,18662.265904,29.75,627.303056,439.112139,460.714286,1,4.0
B-6,3.550605,988.250000,3508.885433,13319.439778,29.75,447.712261,313.398583,460.714286,1,4.0
B-6.1,3.550605,826.333333,2933.983300,11137.158685,29.75,374.358275,262.050793,310.714286,1,6.0
B-6.8,3.550605,905.895833,3216.478314,12209.486463,29.75,410.402906,287.282034,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
B-2,1.866927,1662.437500,3103.649174,25509.655919,29.75,857.467426,600.227198,769.642857,1,2.0
B-2.4,1.866927,2146.791667,4007.902963,32941.940221,29.75,1107.292108,775.104476,921.428571,2,4.0
B-3.7,1.866927,741.187500,1383.742831,11373.322664,20.00,568.666133,398.066293,460.714286,1,4.0
B-4,1.866927,505.291667,943.342570,7753.564603,20.00,387.678230,271.374761,310.714286,1,6.0
B-5,1.866927,770.500000,1438.467123,11823.115086,29.75,397.415633,278.190943,310.714286,1,6.0
B-5.1,1.866927,1384.666667,2585.071352,21247.337256,29.75,714.196210,499.937347,600.000000,1,3.0
B-6,1.866927,988.250000,1844.990441,15164.430219,29.75,509.728747,356.810123,460.714286,1,4.0
B-6.1,1.866927,826.333333,1542.703871,12679.862556,29.75,426.213867,298.349707,310.714286,1,6.0
B-6.8,1.866927,905.895833,1691.241237,13900.727701,29.75,467.251351,327.075946,460.714286,1,4.0


#### East-West Direction

Most likely, the shear walls in the E-W direction will be governed by a rigid diaphragm assumption. 

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ew} through \autoref{tab:shear_wall_forces_level2_ew}.

The walls in the E-W direction are named to be easily found on the plan because there are only 2 lines with walls in this direction. The naming goes `[line B-B or B-C]-[N-S line where the west edge of the wall begins]`. For example, wall "B-C--B-4.6" is the wall on line "B-C" whose left edge begins at "B-4.2".

In [50]:
# Plan dimensions
l_BA__BB = (30 + 9 / 12) * ft
l_BB__BC = (5 + 6 / 12) * ft
l_BC__BD = (30 + 9 / 12) * ft

d_trib_ew = L_floor

In [51]:
# Wall tributary widths and lengths
t_w__BB = l_BA__BB + l_BB__BC / 2
t_w__BC = l_BB__BC / 2 + l_BC__BD

# t_l__BB__1 = 33.75
# t_l__BB__2 = 22.25
# t_l__BB__3 = 16
# t_l__BB__4 = 19
# t_l__BB__5 = 16
# t_l__BB__6 = 34.5
# t_l__BB__7 = 28.25
# t_l__BB__8 = 28.5
# t_l__BB__9 = 31.75
t_l__BB__1 = 33
t_l__BB__2 = 21.5
t_l__BB__3 = 16.75
t_l__BB__4 = 19.5
t_l__BB__5 = 16
t_l__BB__6 = 34
t_l__BB__7 = 28.75
t_l__BB__8 = 28
t_l__BB__9 = 32.5


# t_l__BC__1 = 44
# t_l__BC__2 = 31
# t_l__BC__3 = 27.5
# t_l__BC__4 = 31
# t_l__BC__5 = 15.5
# t_l__BC__6 = 12.34
# t_l__BC__7 = 19
# t_l__BC__8 = 21.33
# t_l__BC__9 = 28.33
t_l__BC__1 = 46.67
t_l__BC__2 = 14
t_l__BC__3 = 21.5
t_l__BC__4 = 20.33
t_l__BC__5 = 30.5
t_l__BC__6 = 16
t_l__BC__7 = 29
t_l__BC__8 = 23.5
t_l__BC__9 = 28.5

In [52]:
# Wall segment lengths
# l_BB__1 = 21
# l_BB__2 = 13.25
# l_BB__3 = 9
# l_BB__4 = 12
# l_BB__5 = 11
# l_BB__6 = 29.75
# l_BB__7 = 12.75
# l_BB__8 = 16.25
# l_BB__9 = 13.75
l_BB__1 = 20
l_BB__2 = 13
l_BB__3 = 9
l_BB__4 = 12
l_BB__5 = 10.67
l_BB__6 = 29
l_BB__7 = 13
l_BB__8 = 11.67
l_BB__9 = 10

# l_BC__1 = 17.25
# l_BC__2 = 23
# l_BC__3 = 22.5
# l_BC__4 = 26
# l_BC__5 = 11.25
# l_BC__6 = 7
# l_BC__7 = 6.75
# l_BC__8 = 10.25
# l_BC__9 = 25.25
l_BC__1 = 16
l_BC__2 = 9.34
l_BC__3 = 16.75
l_BC__4 = 15
l_BC__5 = 26
l_BC__6 = 11.5
l_BC__7 = 16
l_BC__8 = 9.75
l_BC__9 = 24

In [53]:
flexible_ew.loc["Roof", "tributary weight"] = float(f_p__roof)
flexible_ew.loc["Level 4", "tributary weight"] = float(f_p__4)
flexible_ew.loc["Level 3", "tributary weight"] = float(f_p__3)
flexible_ew.loc["Level 2", "tributary weight"] = float(f_p__2)

# Shear wall height is for the floor below
flexible_ew.loc["Roof", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ew.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ew.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ew.loc["Level 2", "wall height"] = float(seismic_loads.loc["Ground", "floor height"])

# Add wall tributary widths and wall lengths to df
wall_tributary = {
    "B-B--B-1": {"trib_width": float(t_w__BB), "trib_depth": t_l__BB__1, "wall_length": l_BB__1},
    "B-B--B-2.5": {"trib_width": float(t_w__BB), "trib_depth": t_l__BB__2, "wall_length": l_BB__2},
    "B-B--B-3.5": {"trib_width": float(t_w__BB), "trib_depth": t_l__BB__3, "wall_length": l_BB__3},
    "B-B--B-4": {"trib_width": float(t_w__BB), "trib_depth": t_l__BB__4, "wall_length": l_BB__4},
    "B-B--B-4.8": {"trib_width": float(t_w__BB), "trib_depth": t_l__BB__5, "wall_length": l_BB__5},
    "B-B--B-5.5": {"trib_width": float(t_w__BB), "trib_depth": t_l__BB__6, "wall_length": l_BB__6},
    "B-B--B-6.5": {"trib_width": float(t_w__BB), "trib_depth": t_l__BB__7, "wall_length": l_BB__7},
    "B-B--B-7.9": {"trib_width": float(t_w__BB), "trib_depth": t_l__BB__8, "wall_length": l_BB__8},
    "B-B--B-8.8": {"trib_width": float(t_w__BB), "trib_depth": t_l__BB__9, "wall_length": l_BB__9},
    "B-C--B-1.8": {"trib_width": float(t_w__BC), "trib_depth": t_l__BC__1, "wall_length": l_BC__1},
    "B-C--B-3.2": {"trib_width": float(t_w__BC), "trib_depth": t_l__BC__2, "wall_length": l_BC__2},
    "B-C--B-3.8": {"trib_width": float(t_w__BC), "trib_depth": t_l__BC__7, "wall_length": l_BC__7},
    "B-C--B-4.6": {"trib_width": float(t_w__BC), "trib_depth": t_l__BC__3, "wall_length": l_BC__3},
    "B-C--B-5.5": {"trib_width": float(t_w__BC), "trib_depth": t_l__BC__4, "wall_length": l_BC__4},
    "B-C--B-6.5": {"trib_width": float(t_w__BC), "trib_depth": t_l__BC__5, "wall_length": l_BC__5},
    "B-C--B-6.9": {"trib_width": float(t_w__BC), "trib_depth": t_l__BC__6, "wall_length": l_BC__6},
    "B-C--B-8.5": {"trib_width": float(t_w__BC), "trib_depth": t_l__BC__8, "wall_length": l_BC__8},
    "B-C--B-9.2": {"trib_width": float(t_w__BC), "trib_depth": t_l__BC__9, "wall_length": l_BC__9},
}

flexible_ew["wall length"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary width"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary depth"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary area"] = flexible_ew["tributary depth"] * flexible_ew["tributary width"]

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ew["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up.\n\
    Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ew["line"] = flexible_ew.index.get_level_values("Wall").str.split("--").str[0]

# Get tributary area and wall length for the shear line
flexible_ew["line tributary area"] = flexible_ew.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ew["line wall length"] = flexible_ew.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ew["line shear demand"] = flexible_ew["line tributary area"] * flexible_ew["tributary weight"]
flexible_ew["line unit shear"] = flexible_ew["line shear demand"] / flexible_ew["line wall length"]
flexible_ew["line cumulative shear demand"] = flexible_ew.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ew["line cumulative unit shear"] = (
    flexible_ew["line cumulative shear demand"] / flexible_ew["line wall length"]
)

flexible_ew["shear demand"] = flexible_ew["line unit shear"] * flexible_ew["wall length"]
flexible_ew["cumulative shear demand"] = flexible_ew.groupby(level="Wall")["shear demand"].cumsum()
flexible_ew["unit shear demand"] = flexible_ew["cumulative shear demand"] / flexible_ew["wall length"]
flexible_ew["adjusted unit shear demand"] = flexible_ew["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ew["sheathed sides"] = 1
flexible_ew["aspect ratio"] = flexible_ew["wall height"] / flexible_ew["wall length"]

#### Check aspect ratios

We will also check the aspect ratios of the E-W walls.

In [54]:
%%render
ratio_max = check_value(flexible_ew["aspect ratio"].max(), 2, "<")

<IPython.core.display.Latex object>

In [55]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [56]:
# using sheathing from the NS calc in memory

flexible_ew[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        flexible_ew[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ew["shear dcr"] > shear_dcr_check_value).any():
    flexible_ew.loc[flexible_ew["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ew.loc[
        flexible_ew["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ew.loc[
            flexible_ew["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

In [57]:
# Update dataframe
flexible.update(flexible_ew)

In [58]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$W_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [59]:
display_table(
    dataframe=flexible_ew,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Roof",
    label="tab:shear_wall_forces_roof_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 4",
    label="tab:shear_wall_forces_level4_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 3",
    label="tab:shear_wall_forces_level3_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 2",
    label="tab:shear_wall_forces_level2_ew",
)

,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
B-B--B-1,4.69743,1105.500,5640.283000,5640.283000,20.00,282.014150,197.409905,310.714286,1,6.0
B-B--B-2.5,4.69743,720.250,3666.183950,3666.183950,13.00,282.014150,197.409905,310.714286,1,6.0
B-B--B-3.5,4.69743,561.125,2538.127350,2538.127350,9.00,282.014150,197.409905,310.714286,1,6.0
B-B--B-4,4.69743,653.250,3384.169800,3384.169800,12.00,282.014150,197.409905,310.714286,1,6.0
B-B--B-4.8,4.69743,536.000,3009.090981,3009.090981,10.67,282.014150,197.409905,310.714286,1,6.0
B-B--B-5.5,4.69743,1139.000,8178.410351,8178.410351,29.00,282.014150,197.409905,310.714286,1,6.0
B-B--B-6.5,4.69743,963.125,3666.183950,3666.183950,13.00,282.014150,197.409905,310.714286,1,6.0
B-B--B-7.9,4.69743,938.000,3291.105131,3291.105131,11.67,282.014150,197.409905,310.714286,1,6.0
B-B--B-8.8,4.69743,1088.750,2820.141500,2820.141500,10.00,282.014150,197.409905,310.714286,1,6.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
B-B--B-1,5.229769,1105.500,6279.471990,11919.754990,20.00,595.987750,417.191425,460.714286,1,4.0
B-B--B-2.5,5.229769,720.250,4081.656793,7747.840744,13.00,595.987750,417.191425,460.714286,1,4.0
B-B--B-3.5,5.229769,561.125,2825.762395,5363.889746,9.00,595.987750,417.191425,460.714286,1,4.0
B-B--B-4,5.229769,653.250,3767.683194,7151.852994,12.00,595.987750,417.191425,460.714286,1,4.0
B-B--B-4.8,5.229769,536.000,3350.098307,6359.189287,10.67,595.987750,417.191425,460.714286,1,4.0
B-B--B-5.5,5.229769,1139.000,9105.234385,17283.644736,29.00,595.987750,417.191425,460.714286,1,4.0
B-B--B-6.5,5.229769,963.125,4081.656793,7747.840744,13.00,595.987750,417.191425,460.714286,1,4.0
B-B--B-7.9,5.229769,938.000,3664.071906,6955.177037,11.67,595.987750,417.191425,460.714286,1,4.0
B-B--B-8.8,5.229769,1088.750,3139.735995,5959.877495,10.00,595.987750,417.191425,460.714286,1,4.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
B-B--B-1,3.550605,1105.500,4263.271288,16183.026278,20.00,809.151314,566.405920,600.0,1,3.0
B-B--B-2.5,3.550605,720.250,2771.126337,10518.967081,13.00,809.151314,566.405920,600.0,1,3.0
B-B--B-3.5,3.550605,561.125,1918.472080,7282.361825,9.00,809.151314,566.405920,600.0,1,3.0
B-B--B-4,3.550605,653.250,2557.962773,9709.815767,12.00,809.151314,566.405920,600.0,1,3.0
B-B--B-4.8,3.550605,536.000,2274.455232,8633.644519,10.67,809.151314,566.405920,600.0,1,3.0
B-B--B-5.5,3.550605,1139.000,6181.743367,23465.388103,29.00,809.151314,566.405920,600.0,1,3.0
B-B--B-6.5,3.550605,963.125,2771.126337,10518.967081,13.00,809.151314,566.405920,600.0,1,3.0
B-B--B-7.9,3.550605,938.000,2487.618796,9442.795833,11.67,809.151314,566.405920,600.0,1,3.0
B-B--B-8.8,3.550605,1088.750,2131.635644,8091.513139,10.00,809.151314,566.405920,600.0,1,3.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
B-B--B-1,1.866927,1105.500,2241.650496,18424.676774,20.00,921.233839,644.863687,769.642857,1,2.0
B-B--B-2.5,1.866927,720.250,1457.072822,11976.039903,13.00,921.233839,644.863687,769.642857,1,2.0
B-B--B-3.5,1.866927,561.125,1008.742723,8291.104548,9.00,921.233839,644.863687,769.642857,1,2.0
B-B--B-4,1.866927,653.250,1344.990298,11054.806064,12.00,921.233839,644.863687,769.642857,1,2.0
B-B--B-4.8,1.866927,536.000,1195.920540,9829.565059,10.67,921.233839,644.863687,769.642857,1,2.0
B-B--B-5.5,1.866927,1139.000,3250.393219,26715.781322,29.00,921.233839,644.863687,769.642857,1,2.0
B-B--B-6.5,1.866927,963.125,1457.072822,11976.039903,13.00,921.233839,644.863687,769.642857,1,2.0
B-B--B-7.9,1.866927,938.000,1308.003064,10750.798898,11.67,921.233839,644.863687,769.642857,1,2.0
B-B--B-8.8,1.866927,1088.750,1120.825248,9212.338387,10.00,921.233839,644.863687,769.642857,1,2.0


### Rigid Diaphragm Assumption

With a rigid diaphragm assumption, we now must calculate the following in each direction:

1. The relative stiffnesses of the shear walls to obtain a center of rigidity.
2. The center of mass of the diaphragm with the center of rigidity to obtain an eccentricity.

The relative stiffnesses of the shear walls can be calculated by using the deflection equation from NDS SDPWS Sec. 4.3.4

\begin{align*}
k &= \frac{F}{\delta},\, \text{assume }F = \text{induced load demand in the shear wall} \\
\delta_{sw} &= \frac{8vh^3}{EAb} + \frac{vh}{1000 G_a} + \frac{h \Delta_A}{b}\quad \text{(SDPWS Eq. 4.3-1)}
\end{align*}

where

- $b =$ the shear wall length [ft]
- $\Delta_A =$ vertical deformation of the wall overturning anchorage system + vertical compression deformation [in]
- $E =$ modulus of elasticity of end posts [psi]
- $A =$ area of end post cross-section [in$^2$]
- $G_a =$ apparent shear wall shear stiffness from nail slip and panel shear deformation [kip/in]
- $h =$ shear wall height [ft]
- $v =$ unit shear force induced by the design load [plf]
- $\delta_{sw} =$ maximum shear wall deflection determined by elastic analysis [in]

Note that for the following calculations, we will conservatively use the unit shear force induced by the nominal load, not the design load, for the value of $v$. This is because the stiffnesses obtained from this use will allow us to use the stiffnesses for deflection limits and story drift calculations

We will assume that the end posts are (2) 2x4 Douglas Fir No. 2; this means:

In [60]:
%%render
E_endpost = 1_600_000 * psi
A_endpost = 2 * 5.25 * inch**2

<IPython.core.display.Latex object>

We will assume a continuous tiedown rod system. A conservative first-pass value for $\Delta_A$ can be:

In [61]:
%%render params
Delta_A = 0.25 * inch

<IPython.core.display.Latex object>

#### Shear Wall Rigidities

With the deflections of each shear wall, along with the force demand of each shear wall, we can obtain the stiffnesses (and therefore, relative stiffnesses) of all the walls.

Each relative stiffness is relative to the level. The sum of all relative stiffnesses on each level is $1.0$.

\begin{align*}
k &= \frac{F_x}{\delta_{sw}} \\
R &= \frac{k_{wall}}{\sum {k_{wall}}}
\end{align*}

##### North-South Direction

Rigidities in the N-S direction can be found in \autoref{tab:wall_rigidities_ns_roof} through \autoref{tab:wall_rigidities_ns_level2}.

In [62]:
rigid_ns["unit shear demand"] = flexible_ns["unit shear demand"]
rigid_ns["wall length"] = flexible_ns["wall length"]
rigid_ns["wall height"] = flexible_ns["wall height"]
rigid_ns["shear stiffness"] = flexible_ns["shear stiffness"]
rigid_ns["nail spacing"] = flexible_ns["nail spacing"]
rigid_ns["end post youngs modulus"] = float(E_endpost)
rigid_ns["end post area"] = float(A_endpost)
rigid_ns["Delta_A"] = float(Delta_A)

rigid_ns["delta_sw"] = (
    8
    * rigid_ns["unit shear demand"]
    * rigid_ns["wall height"] ** 3
    / (rigid_ns["end post youngs modulus"] * rigid_ns["end post area"] * rigid_ns["wall length"])
    + rigid_ns["unit shear demand"] * rigid_ns["wall height"] / (1000 * rigid_ns["shear stiffness"])
    + rigid_ns["wall height"] * rigid_ns["Delta_A"] / rigid_ns["wall length"]
)

In [63]:
rigid_ns["cumulative shear demand"] = flexible_ns["cumulative shear demand"]
rigid_ns["wall stiffness"] = rigid_ns["cumulative shear demand"] / rigid_ns["delta_sw"]
rigid_ns["total level stiffness"] = rigid_ns.groupby("Level")["wall stiffness"].transform("sum")
rigid_ns["total level cumulative shear"] = rigid_ns.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ns["relative wall stiffness"] = rigid_ns["wall stiffness"] / rigid_ns["total level stiffness"]
rigid_ns["direct shear force"] = rigid_ns["total level cumulative shear"] * rigid_ns["relative wall stiffness"]

In [64]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [65]:
display_table(
    dataframe=rigid_ns,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Roof",
    label="tab:wall_rigidities_ns_roof",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 4",
    label="tab:wall_rigidities_ns_level4",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 3",
    label="tab:wall_rigidities_ns_level3",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 2",
    label="tab:wall_rigidities_ns_level2",
)

,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
B-2,7809.183325,0.275731,28321.788143,0.088768
B-2.4,10084.402984,0.331582,30413.012714,0.095323
B-3.7,3481.676193,0.253490,13734.941315,0.043049
B-4,2373.572094,0.212596,11164.701275,0.034993
B-5,3619.369601,0.172881,20935.663907,0.065618
B-5.1,6504.374356,0.243701,26690.020201,0.083654
B-6,4642.234923,0.197989,23446.874544,0.073489
B-6.1,3881.642761,0.179319,21646.607203,0.067847
B-6.8,4255.382013,0.188493,22575.788363,0.070759


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
B-2,16503.347775,0.441144,37410.320197,0.090486
B-2.4,21311.627941,0.497391,42846.864585,0.103635
B-3.7,7357.915758,0.417312,17631.691186,0.042646
B-4,5016.130894,0.326268,15374.276740,0.037186
B-5,7648.906777,0.285829,26760.438609,0.064726
B-5.1,13745.861455,0.382178,35967.176868,0.086995
B-6,9810.554345,0.341671,28713.489272,0.069450
B-6.1,8203.175384,0.300147,27330.500577,0.066105
B-6.8,8993.008149,0.320551,28054.846437,0.067857


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
B-2,22406.006745,0.518401,43221.359907,0.091213
B-2.4,28934.037258,0.550257,52582.763713,0.110969
B-3.7,9989.579834,0.453519,22026.835054,0.046485
B-4,6810.222033,0.396019,17196.719905,0.036291
B-5,10384.647963,0.356501,29129.358340,0.061474
B-5.1,18662.265904,0.487311,38296.438900,0.080819
B-6,13319.439778,0.373059,35703.277635,0.075347
B-6.1,11137.158685,0.375941,29624.779890,0.062519
B-6.8,12209.486463,0.403642,30248.305465,0.063835


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
B-2,25509.655919,0.546246,46699.912665,0.103352
B-2.4,32941.940221,0.872645,37749.539303,0.083544
B-3.7,11373.322664,0.549028,20715.375977,0.045846
B-4,7753.564603,0.476238,16280.850199,0.036031
B-5,11823.115086,0.432762,27320.145342,0.060463
B-5.1,21247.337256,0.546302,38893.059431,0.086075
B-6,15164.430219,0.453864,33411.847557,0.073944
B-6.1,12679.862556,0.457119,27738.674731,0.061389
B-6.8,13900.727701,0.424095,32777.383720,0.072540


##### East-West Direction

Rigidities in the E-W direction can be found in \autoref{tab:wall_rigidities_ew_roof}, \autoref{tab:wall_rigidities_ew_level4}, \autoref{tab:wall_rigidities_ew_level3}, \autoref{tab:wall_rigidities_ew_level2},

In [66]:
rigid_ew["unit shear demand"] = flexible_ew["unit shear demand"]
rigid_ew["wall length"] = flexible_ew["wall length"]
rigid_ew["wall height"] = flexible_ew["wall height"]
rigid_ew["shear stiffness"] = flexible_ew["shear stiffness"]
rigid_ew["nail spacing"] = flexible_ew["nail spacing"]
rigid_ew["end post youngs modulus"] = float(E_endpost)
rigid_ew["end post area"] = float(A_endpost)
rigid_ew["Delta_A"] = float(Delta_A)

rigid_ew["delta_sw"] = (
    8
    * rigid_ew["unit shear demand"]
    * rigid_ew["wall height"] ** 3
    / (rigid_ew["end post youngs modulus"] * rigid_ew["end post area"] * rigid_ew["wall length"])
    + rigid_ew["unit shear demand"] * rigid_ew["wall height"] / (1000 * rigid_ew["shear stiffness"])
    + rigid_ew["wall height"] * rigid_ew["Delta_A"] / rigid_ew["wall length"]
)

In [67]:
rigid_ew["cumulative shear demand"] = flexible_ew["cumulative shear demand"]
rigid_ew["wall stiffness"] = rigid_ew["cumulative shear demand"] / rigid_ew["delta_sw"]
rigid_ew["total level stiffness"] = rigid_ew.groupby("Level")["wall stiffness"].transform("sum")
rigid_ew["total level cumulative shear"] = rigid_ew.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ew["relative wall stiffness"] = rigid_ew["wall stiffness"] / rigid_ew["total level stiffness"]
rigid_ew["direct shear force"] = rigid_ew["total level cumulative shear"] * rigid_ew["relative wall stiffness"]

In [68]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [69]:
display_table(
    dataframe=rigid_ew,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Roof",
    label="tab:wall_rigidities_ew_roof",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 4",
    label="tab:wall_rigidities_ew_level4",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 3",
    label="tab:wall_rigidities_ew_level3",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 2",
    label="tab:wall_rigidities_ew_level2",
)

,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
B-B--B-1,5640.283000,0.333153,16929.992829,0.082548
B-B--B-2.5,3666.183950,0.404077,9072.993381,0.044239
B-B--B-3.5,2538.127350,0.494138,5136.476394,0.025045
B-B--B-4,3384.169800,0.420963,8039.113655,0.039198
B-B--B-4.8,3009.090981,0.448326,6711.830239,0.032726
B-B--B-5.5,8178.410351,0.292276,27981.772622,0.136435
B-B--B-6.5,3666.183950,0.404077,9072.993381,0.044239
B-B--B-7.9,3291.105131,0.427171,7704.426562,0.037566
B-B--B-8.8,2820.141500,0.464868,6066.543540,0.029580


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
B-B--B-1,11919.754990,0.515787,23109.840212,0.080334
B-B--B-2.5,7747.840744,0.595305,13014.902351,0.045242
B-B--B-3.5,5363.889746,0.696281,7703.627945,0.026779
B-B--B-4,7151.852994,0.614238,11643.450708,0.040475
B-B--B-4.8,6359.189287,0.644918,9860.465004,0.034277
B-B--B-5.5,17283.644736,0.469956,36777.137405,0.127844
B-B--B-6.5,7747.840744,0.595305,13014.902351,0.045242
B-B--B-7.9,6955.177037,0.621198,11196.389718,0.038921
B-B--B-8.8,5959.877495,0.663464,8982.971715,0.031226


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
B-B--B-1,16183.026278,0.600715,26939.616342,0.079646
B-B--B-2.5,10518.967081,0.683397,15392.181375,0.045506
B-B--B-3.5,7282.361825,0.788390,9237.007202,0.027309
B-B--B-4,9709.815767,0.703083,13810.341546,0.040830
B-B--B-4.8,8633.644519,0.734983,11746.725841,0.034729
B-B--B-5.5,23465.388103,0.553061,42428.235817,0.125437
B-B--B-6.5,10518.967081,0.683397,15392.181375,0.045506
B-B--B-7.9,9442.795833,0.710320,13293.725022,0.039302
B-B--B-8.8,8091.513139,0.754267,10727.650514,0.031716


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
B-B--B-1,18424.676774,0.637726,28891.211408,0.085656
B-B--B-2.5,11976.039903,0.733092,16336.330003,0.048434
B-B--B-3.5,8291.104548,0.854193,9706.363597,0.028777
B-B--B-4,11054.806064,0.755799,14626.653488,0.043365
B-B--B-4.8,9829.565059,0.792593,12401.785873,0.036769
B-B--B-5.5,26715.781322,0.582761,45843.452206,0.135916
B-B--B-6.5,11976.039903,0.733092,16336.330003,0.048434
B-B--B-7.9,10750.798898,0.764146,14069.041976,0.041712
B-B--B-8.8,9212.338387,0.814835,11305.770379,0.033519


#### Center of Rigidity and Center of Mass

We can find the center of rigidity by taking the weighted sum of all wall stiffnesses and their locations relative to a datum. In this case, we will use the north-west-most corner of the building (i.e., the intersection of lines B-1 and B-A).

$$
\bar{x}_r = \frac{\sum k_{ns} x}{\sum k_{ns}},\, \bar{y}_r = \frac{\sum k_{ew} y}{\sum k_{ew}}
$$

Note: each wall is assumed to have zero rigidity in the direction perpendicular to them.

The building is symmetric in each direction when projected to that direction, so the centers of mass are just half of each total plan dimension.

In [70]:
%%render
X_plan = L_floor + L_jog
Y_plan = w_building
CM_x = X_plan / 2
CM_y = Y_plan / 2

<IPython.core.display.Latex object>

An accidental torsion eccentricity of 5% must be added/subtracted.

In [71]:
%%render
CM_x__a__plus = CM_x + X_plan * 0.05
CM_x__a__minus = CM_x - X_plan * 0.05
CM_y__a__plus = CM_y + Y_plan * 0.05
CM_y__a__minus = CM_y - Y_plan * 0.05

<IPython.core.display.Latex object>

The eccentricity used is determined by the location of the wall in relation to the center of rigidity. Since the wall rotates about the center of rigidity, we want to position the center of mass to ensure each wall is getting the maximum possible force to be conservative. If the center of mass is positioned on the opposite side of the center of rigidity relative to the wall in question, that mass will _decrease_ the shear experienced by that wall.

This means we always want the center of mass to be on the same side of the center of rigidity as the wall. The eccentricities shown in the following tables are based on the worst-case scenario. This is not immediately obvious due to all torques being given a positive value. 

How the center of rigidity, center of mass, and eccentricity interact are illustrated in \autoref{fig:rigid_diaphragm_moments}. The moments generated by the rotation of the center of mass around the center of rigidity differ based on the chosen eccentricity.

![Moments applied by a rigid diaphragm](images/rigid_diaphragm_moments.png){width=50% #fig:rigid_diaphragm_moments}

$$
d_x = \left| x - \text{CR}_x \right|\text{ and }\, d_y = \left| y - \text{CR}_y \right|
$$

In [72]:
# Update dataframe
rigid = rigid.reindex(columns=rigid_ns.columns)
rigid.update(rigid_ns)
rigid.update(rigid_ew)

In [ ]:
# Location of wall lines of rigidity
wall_rigidity_loc = {
    # N-S
    "B-2": {"x": float(l_B1__B2), "y": 0},
    "B-2.4": {"x": float(l_B1__B24), "y": 0},
    "B-3.7": {"x": float(l_B1__B2 + l_B2__B3 + l_B3__B37), "y": 0},
    "B-4": {"x": float(l_B1__B2 + l_B2__B3 + l_B3__B4), "y": 0},
    "B-5": {"x": float(l_B1__B2 + l_B2__B3 + l_B3__B4 + l_B4__B5), "y": 0},
    "B-5.1": {"x": float(l_B1__B24 + l_B24__B35 + l_B35__B51), "y": 0},
    "B-6": {"x": float(l_B1__B2 + l_B2__B3 + l_B3__B4 + l_B4__B5 + l_B5__B6), "y": 0},
    "B-6.1": {"x": float(l_B1__B24 + l_B24__B35 + l_B35__B51 + l_B51__B61), "y": 0},
    "B-6.8": {"x": float(l_B1__B24 + l_B24__B35 + l_B35__B51 + l_B51__B61 + l_B61__B68), "y": 0},
    "B-7": {"x": float(l_B1__B2 + l_B2__B3 + l_B3__B4 + l_B4__B5 + l_B5__B6 + l_B6__B7), "y": 0},
    "B-7.8": {"x": float(l_B1__B24 + l_B24__B35 + l_B35__B51 + l_B51__B61 + l_B61__B68 + l_B68__B78), "y": 0},
    "B-8": {"x": float(l_B1__B2 + l_B2__B3 + l_B3__B4 + l_B4__B5 + l_B5__B6 + l_B6__B7 + l_B7__B8), "y": 0},
    "B-8.2": {
        "x": float(l_B1__B24 + l_B24__B35 + l_B35__B51 + l_B51__B61 + l_B61__B68 + l_B68__B78 + l_B78__B82),
        "y": 0,
    },
    "B-8.9": {
        "x": float(
            l_B1__B24 + l_B24__B35 + l_B35__B51 + l_B51__B61 + l_B61__B68 + l_B68__B78 + l_B78__B82 + l_B82__B89
        ),
        "y": 0,
    },
    "B-9": {"x": float(l_B1__B2 + l_B2__B3 + l_B3__B4 + l_B4__B5 + l_B5__B6 + l_B6__B7 + l_B7__B8 + l_B8__B9), "y": 0},
    # E-W
    "B-B--B-1": {"x": 0, "y": float(l_BA__BB)},
    "B-B--B-2.5": {"x": 0, "y": float(l_BA__BB)},
    "B-B--B-3.5": {"x": 0, "y": float(l_BA__BB)},
    "B-B--B-4": {"x": 0, "y": float(l_BA__BB)},
    "B-B--B-4.8": {"x": 0, "y": float(l_BA__BB)},
    "B-B--B-5.5": {"x": 0, "y": float(l_BA__BB)},
    "B-B--B-6.5": {"x": 0, "y": float(l_BA__BB)},
    "B-B--B-7.9": {"x": 0, "y": float(l_BA__BB)},
    "B-B--B-8.8": {"x": 0, "y": float(l_BA__BB)},
    "B-C--B-1.8": {"x": 0, "y": float(l_BA__BB + l_BB__BC)},
    "B-C--B-3.2": {"x": 0, "y": float(l_BA__BB + l_BB__BC)},
    "B-C--B-3.8": {"x": 0, "y": float(l_BA__BB + l_BB__BC)},
    "B-C--B-4.6": {"x": 0, "y": float(l_BA__BB + l_BB__BC)},
    "B-C--B-5.5": {"x": 0, "y": float(l_BA__BB + l_BB__BC)},
    "B-C--B-6.5": {"x": 0, "y": float(l_BA__BB + l_BB__BC)},
    "B-C--B-6.9": {"x": 0, "y": float(l_BA__BB + l_BB__BC)},
    "B-C--B-8.5": {"x": 0, "y": float(l_BA__BB + l_BB__BC)},
    "B-C--B-9.2": {"x": 0, "y": float(l_BA__BB + l_BB__BC)},
}

In [74]:
# Calc centers of rigidity
rigid["x"] = rigid.index.get_level_values("Wall").map({key: value["x"] for key, value in wall_rigidity_loc.items()})
rigid["y"] = rigid.index.get_level_values("Wall").map({key: value["y"] for key, value in wall_rigidity_loc.items()})

rigid["kx"] = rigid["relative wall stiffness"] * rigid["x"]
rigid["CRx"] = rigid.groupby(["Level", "Direction"])["kx"].transform("sum")  # all relative wall rigidities sum to 1
rigid["ky"] = rigid["relative wall stiffness"] * rigid["y"]
rigid["CRy"] = rigid.groupby(["Level", "Direction"])["ky"].transform("sum")  # all relative wall rigidities sum to 1

# x eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
plan_dim_x = float(X_plan)
CMx = plan_dim_x / 2
CMxa_plus = CMx + plan_dim_x * 0.05
CMxa_minus = CMx - plan_dim_x * 0.05
rigid["ex"] = np.where(rigid["x"] <= rigid["CRx"], rigid["CRx"] - CMxa_minus, CMxa_plus - rigid["CRx"])
rigid["ex"] = np.where(rigid["ex"] <= 0, 0, rigid["ex"])  # assign 0s to negatives

# y eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
plan_dim_y = float(w_building)
CMy = plan_dim_y / 2
CMya_plus = CMy + plan_dim_y * 0.05
CMya_minus = CMy - plan_dim_y * 0.05
rigid["ey"] = np.where(rigid["y"] <= rigid["CRy"], rigid["CRy"] - CMya_minus, CMya_plus - rigid["CRy"])
rigid["ey"] = np.where(rigid["ey"] <= 0, 0, rigid["ey"])  # assign 0s to negatives

# distance from CR
rigid["dx"] = abs(rigid["x"] - rigid["CRx"])
rigid["dy"] = abs(rigid["y"] - rigid["CRy"])

# Torque
rigid["torque x"] = rigid["total level cumulative shear"] * rigid["ex"]
rigid["torque y"] = rigid["total level cumulative shear"] * rigid["ey"]

# Polar moment of inertia
rigid["Jx wall"] = rigid["relative wall stiffness"] * rigid["dx"] ** 2
rigid["Jy wall"] = rigid["relative wall stiffness"] * rigid["dy"] ** 2
rigid["J"] = (rigid["Jx wall"] + rigid["Jy wall"]).groupby(["Level"]).transform("sum")

rigid["torsional shear force x"] = rigid["torque x"] * rigid["relative wall stiffness"] * rigid["dx"] / rigid["J"]
rigid["torsional shear force y"] = rigid["torque y"] * rigid["relative wall stiffness"] * rigid["dy"] / rigid["J"]
rigid["torsional shear force"] = np.where(
    rigid["torsional shear force x"] == 0, rigid["torsional shear force y"], rigid["torsional shear force x"]
)
rigid["shear force demand"] = rigid["torsional shear force"] + rigid["direct shear force"]

The centers of rigidity for each floor are given in \autoref{tab:center_of_rigidity_roof} through \autoref{tab:center_of_rigidity_level2}.

In [75]:
column_name_map = {
    "x": "$x$ [ft]",
    "y": "$y$ [ft]",
    "CRx": "CR$_x$ [ft]",
    "dx": "$d_x$ [ft]",
    "ex": "e$_x$ [ft]",
    "CRy": "CR$_y$ [ft]",
    "dy": "$d_y$ [ft]",
    "ey": "e$_y$ [ft]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Roof",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Roof",
    label="tab:center_of_rigidity_roof",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 4",
    label="tab:center_of_rigidity_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 3",
    label="tab:center_of_rigidity_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 2",
    label="tab:center_of_rigidity_level2",
)

,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
B-2,31.916667,0.0,124.68853,92.771864,18.93853,0.0,0.0,0.00
B-2.4,40.083333,0.0,124.68853,84.605197,18.93853,0.0,0.0,0.00
B-3.7,67.333333,0.0,124.68853,57.355197,18.93853,0.0,0.0,0.00
B-4,76.166667,0.0,124.68853,48.521864,18.93853,0.0,0.0,0.00
B-5,97.500000,0.0,124.68853,27.188530,18.93853,0.0,0.0,0.00
B-5.1,98.083333,0.0,124.68853,26.605197,18.93853,0.0,0.0,0.00
B-6,122.166667,0.0,124.68853,2.521864,18.93853,0.0,0.0,0.00
B-6.1,122.750000,0.0,124.68853,1.938530,18.93853,0.0,0.0,0.00
B-6.8,147.416667,0.0,124.68853,22.728136,4.56147,0.0,0.0,0.00


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
B-2,31.916667,0.0,123.345033,91.428366,17.595033,0.0,0.0,0.00
B-2.4,40.083333,0.0,123.345033,83.261700,17.595033,0.0,0.0,0.00
B-3.7,67.333333,0.0,123.345033,56.011700,17.595033,0.0,0.0,0.00
B-4,76.166667,0.0,123.345033,47.178366,17.595033,0.0,0.0,0.00
B-5,97.500000,0.0,123.345033,25.845033,17.595033,0.0,0.0,0.00
B-5.1,98.083333,0.0,123.345033,25.261700,17.595033,0.0,0.0,0.00
B-6,122.166667,0.0,123.345033,1.178366,17.595033,0.0,0.0,0.00
B-6.1,122.750000,0.0,123.345033,0.595033,17.595033,0.0,0.0,0.00
B-6.8,147.416667,0.0,123.345033,24.071634,5.904967,0.0,0.0,0.00


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
B-2,31.916667,0.0,122.493529,90.576863,16.743529,0.0,0.0,0.00
B-2.4,40.083333,0.0,122.493529,82.410196,16.743529,0.0,0.0,0.00
B-3.7,67.333333,0.0,122.493529,55.160196,16.743529,0.0,0.0,0.00
B-4,76.166667,0.0,122.493529,46.326863,16.743529,0.0,0.0,0.00
B-5,97.500000,0.0,122.493529,24.993529,16.743529,0.0,0.0,0.00
B-5.1,98.083333,0.0,122.493529,24.410196,16.743529,0.0,0.0,0.00
B-6,122.166667,0.0,122.493529,0.326863,16.743529,0.0,0.0,0.00
B-6.1,122.750000,0.0,122.493529,0.256471,6.756471,0.0,0.0,0.00
B-6.8,147.416667,0.0,122.493529,24.923137,6.756471,0.0,0.0,0.00


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
B-2,31.916667,0.0,124.241315,92.324648,18.491315,0.0,0.0,0.00
B-2.4,40.083333,0.0,124.241315,84.157982,18.491315,0.0,0.0,0.00
B-3.7,67.333333,0.0,124.241315,56.907982,18.491315,0.0,0.0,0.00
B-4,76.166667,0.0,124.241315,48.074648,18.491315,0.0,0.0,0.00
B-5,97.500000,0.0,124.241315,26.741315,18.491315,0.0,0.0,0.00
B-5.1,98.083333,0.0,124.241315,26.157982,18.491315,0.0,0.0,0.00
B-6,122.166667,0.0,124.241315,2.074648,18.491315,0.0,0.0,0.00
B-6.1,122.750000,0.0,124.241315,1.491315,18.491315,0.0,0.0,0.00
B-6.8,147.416667,0.0,124.241315,23.175352,5.008685,0.0,0.0,0.00


And now, we can get the forces in the shear walls using the rigid diaphragm assumption in \autoref{tab:rigid_forces_roof} through \autoref{tab:rigid_forces_level2}. These forces can be compared to the flexible shear wall forces obtained earlier, $F_{flex}$.

In [76]:
rigid["flexible shear force demand"] = flexible["cumulative shear demand"]

In [77]:
column_name_map = {
    "torque x": "$T_x$ [lbf-ft]",
    "torque y": "$T_y$ [lbf-ft]",
    "dx": "$d_x$ [ft]",
    "dy": "$d_y$ [ft]",
    "relative wall stiffness": "$R$ [-]",
    "torsional shear force": "$F_t$ [lbf]",
    "direct shear force": "$F_v$ [lbf]",
    "shear force demand": "$F_{tot}$ [lbf]",
    "flexible shear force demand": "$F_{flex}$ [lbf]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Roof",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Roof",
    label="tab:rigid_forces_roof",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 4",
    label="tab:rigid_forces_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 3",
    label="tab:rigid_forces_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 2",
    label="tab:rigid_forces_level2",
)

,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
B-2,1.370911e+06,0.000000e+00,92.771864,0.0,0.088768,3356.831794,6425.711786,9782.543580,7809.183325
B-2.4,1.370911e+06,0.000000e+00,84.605197,0.0,0.095323,3287.373982,6900.173579,10187.547560,10084.402984
B-3.7,1.370911e+06,0.000000e+00,57.355197,0.0,0.043049,1006.450020,3116.214762,4122.664781,3481.676193
B-4,1.370911e+06,0.000000e+00,48.521864,0.0,0.034993,692.113323,2533.072849,3225.186172,2373.572094
B-5,1.370911e+06,0.000000e+00,27.188530,0.0,0.065618,727.218801,4749.931100,5477.149901,3619.369601
B-5.1,1.370911e+06,0.000000e+00,26.605197,0.0,0.083654,907.210377,6055.492559,6962.702937,6504.374356
B-6,1.370911e+06,0.000000e+00,2.521864,0.0,0.073489,75.543861,5319.680288,5395.224149,4642.234923
B-6.1,1.370911e+06,0.000000e+00,1.938530,0.0,0.067847,53.611140,4911.231534,4964.842674,3881.642761
B-6.8,3.301929e+05,0.000000e+00,22.728136,0.0,0.070759,157.891195,5122.046272,5279.937467,4255.382013


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
B-2,2.691655e+06,0.000000e+00,91.428366,0.0,0.090486,6508.442572,13842.326243,20350.768815,16503.347775
B-2.4,2.691655e+06,0.000000e+00,83.261700,0.0,0.103635,6788.424795,15853.921457,22642.346251,21311.627941
B-3.7,2.691655e+06,0.000000e+00,56.011700,0.0,0.042646,1879.219082,6523.965054,8403.184136,7357.915758
B-4,2.691655e+06,0.000000e+00,47.178366,0.0,0.037186,1380.200829,5688.691069,7068.891898,5016.130894
B-5,2.691655e+06,0.000000e+00,25.845033,0.0,0.064726,1316.057938,9901.725505,11217.783442,7648.906777
B-5.1,2.691655e+06,0.000000e+00,25.261700,0.0,0.086995,1728.914866,13308.343623,15037.258489,13745.861455
B-6,2.691655e+06,0.000000e+00,1.178366,0.0,0.069450,64.382961,10624.380758,10688.763719,9810.554345
B-6.1,2.691655e+06,0.000000e+00,0.595033,0.0,0.066105,30.945196,10112.656170,10143.601367,8203.175384
B-6.8,9.033309e+05,0.000000e+00,24.071634,0.0,0.067857,431.266334,10380.673970,10811.940304,8993.008149


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
B-2,3.477513e+06,0.000000e+00,90.576863,0.0,0.091213,8289.043592,18944.249614,27233.293206,22406.006745
B-2.4,3.477513e+06,0.000000e+00,82.410196,0.0,0.110969,9175.147994,23047.423850,32222.571845,28934.037258
B-3.7,3.477513e+06,0.000000e+00,55.160196,0.0,0.046485,2572.566577,9654.528741,12227.095319,9989.579834
B-4,3.477513e+06,0.000000e+00,46.326863,0.0,0.036291,1686.814131,7537.452665,9224.266796,6810.222033
B-5,3.477513e+06,0.000000e+00,24.993529,0.0,0.061474,1541.513132,12767.618527,14309.131658,10384.647963
B-5.1,3.477513e+06,0.000000e+00,24.410196,0.0,0.080819,1979.330796,16785.619412,18764.950208,18662.265904
B-6,3.477513e+06,0.000000e+00,0.326863,0.0,0.075347,24.709402,15649.017176,15673.726578,13319.439778
B-6.1,1.403271e+06,0.000000e+00,0.256471,0.0,0.062519,6.491638,12984.765547,12991.257185,11137.158685
B-6.8,1.403271e+06,0.000000e+00,24.923137,0.0,0.063835,644.117891,13258.061532,13902.179422,12209.486463


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
B-2,4.372499e+06,0.000000e+00,92.324648,0.0,0.103352,12225.345687,24438.914723,36664.260410,25509.655919
B-2.4,4.372499e+06,0.000000e+00,84.157982,0.0,0.083544,9008.124606,19755.021353,28763.145959,32941.940221
B-3.7,4.372499e+06,0.000000e+00,56.907982,0.0,0.045846,3342.669362,10840.733485,14183.402847,11373.322664
B-4,4.372499e+06,0.000000e+00,48.074648,0.0,0.036031,2219.323513,8520.065391,10739.388904,7753.564603
B-5,4.372499e+06,0.000000e+00,26.741315,0.0,0.060463,2071.539332,14297.129569,16368.668901,11823.115086
B-5.1,4.372499e+06,0.000000e+00,26.157982,0.0,0.086075,2884.720605,20353.446260,23238.166865,21247.337256
B-6,4.372499e+06,0.000000e+00,2.074648,0.0,0.073944,196.549711,17485.028271,17681.577983,15164.430219
B-6.1,4.372499e+06,0.000000e+00,1.491315,0.0,0.061389,117.295817,14516.153620,14633.449438,12679.862556
B-6.8,1.184365e+06,0.000000e+00,23.175352,0.0,0.072540,583.423545,17153.001791,17736.425337,13900.727701


### Final Shear Wall Nailing Selection

With the envelope procedure complete, we can now select the shear wall sheathing and nailing specifications in \autoref{tab:shear_walls_roof} through \autoref{tab:shear_walls_level2}.

In [78]:
shear_walls["wall length"] = rigid["wall length"]
shear_walls["rigid shear force demand"] = rigid["shear force demand"]
shear_walls["flexible shear force demand"] = rigid["flexible shear force demand"]
shear_walls["shear force demand"] = np.where(
    shear_walls["flexible shear force demand"] >= shear_walls["rigid shear force demand"],
    shear_walls["flexible shear force demand"],
    shear_walls["rigid shear force demand"],
)
shear_walls["unit shear demand"] = shear_walls["shear force demand"] / shear_walls["wall length"]
shear_walls["adjusted unit shear demand"] = (
    shear_walls["shear force demand"] / shear_walls["wall length"] * LOAD_FACTOR_SEISMIC_ASD
)
shear_walls["sheathed sides"] = 1

# using sheathing from the NS calc in memory

shear_walls[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        shear_walls[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (shear_walls["shear dcr"] > shear_dcr_check_value).any():
    shear_walls.loc[shear_walls["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    shear_walls.loc[
        shear_walls["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        shear_walls.loc[
            shear_walls["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

column_name_map = {
    "shear force demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Roof",
    position_float="centering",
    caption="Final shear wall design values - Roof",
    label="tab:shear_walls_roof",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    caption="Final shear wall design values - Level 4",
    label="tab:shear_walls_level4",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    caption="Final shear wall design values - Level 3",
    label="tab:shear_walls_level3",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    caption="Final shear wall design values - Level 2",
    label="tab:shear_walls_level2",
)

,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
B-2,9782.543580,29.75,328.824994,230.177496,310.714286,1,6.0
B-2.4,10187.547560,29.75,342.438573,239.707001,310.714286,1,6.0
B-3.7,4122.664781,20.00,206.133239,144.293267,310.714286,1,6.0
B-4,3225.186172,20.00,161.259309,112.881516,310.714286,1,6.0
B-5,5477.149901,29.75,184.105879,128.874115,310.714286,1,6.0
B-5.1,6962.702937,29.75,234.040435,163.828304,310.714286,1,6.0
B-6,5395.224149,29.75,181.352072,126.946451,310.714286,1,6.0
B-6.1,4964.842674,29.75,166.885468,116.819828,310.714286,1,6.0
B-6.8,5279.937467,29.75,177.476890,124.233823,310.714286,1,6.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
B-2,20350.768815,29.75,684.059456,478.841619,600.000000,1,3.0
B-2.4,22642.346251,29.75,761.087269,532.761088,600.000000,1,3.0
B-3.7,8403.184136,20.00,420.159207,294.111445,310.714286,1,6.0
B-4,7068.891898,20.00,353.444595,247.411216,310.714286,1,6.0
B-5,11217.783442,29.75,377.068351,263.947846,310.714286,1,6.0
B-5.1,15037.258489,29.75,505.454067,353.817847,460.714286,1,4.0
B-6,10688.763719,29.75,359.286175,251.500323,310.714286,1,6.0
B-6.1,10143.601367,29.75,340.961390,238.672973,310.714286,1,6.0
B-6.8,10811.940304,29.75,363.426565,254.398595,310.714286,1,6.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
B-2,27233.293206,29.75,915.404814,640.783370,769.642857,1,2.0
B-2.4,32222.571845,29.75,1083.111659,758.178161,769.642857,1,2.0
B-3.7,12227.095319,20.00,611.354766,427.948336,460.714286,1,4.0
B-4,9224.266796,20.00,461.213340,322.849338,460.714286,1,4.0
B-5,14309.131658,29.75,480.979215,336.685451,460.714286,1,4.0
B-5.1,18764.950208,29.75,630.754629,441.528240,460.714286,1,4.0
B-6,15673.726578,29.75,526.847952,368.793567,460.714286,1,4.0
B-6.1,12991.257185,29.75,436.680914,305.676640,310.714286,1,6.0
B-6.8,13902.179422,29.75,467.300149,327.110104,460.714286,1,4.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
B-2,36664.260410,29.75,1232.412115,862.688480,921.428571,2,4.0
B-2.4,32941.940221,29.75,1107.292108,775.104476,921.428571,2,4.0
B-3.7,14183.402847,20.00,709.170142,496.419100,600.000000,1,3.0
B-4,10739.388904,20.00,536.969445,375.878612,460.714286,1,4.0
B-5,16368.668901,29.75,550.207358,385.145151,460.714286,1,4.0
B-5.1,23238.166865,29.75,781.114853,546.780397,600.000000,1,3.0
B-6,17681.577983,29.75,594.338756,416.037129,460.714286,1,4.0
B-6.1,14633.449438,29.75,491.880653,344.316457,460.714286,1,4.0
B-6.8,17736.425337,29.75,596.182364,417.327655,460.714286,1,4.0


### Shear Wall Holddown Forces

## Diaphragm Design

We must first get the diaphragm inertial design forces. These are distinct from the seismic forces determined in \autoref{tab:seismic_forces_calculation_table}.

\begin{align*}
F_{px} &= \frac{\sum_{i=x}^n F_i}{\sum_{i=x}^n w_i} w_{px} \quad \text{(ASCE Eq. 12.10-1)} \\
F_{px} &\ge 0.2 S_{DS} I_e w_{px} \quad \text{(ASCE Eq. 12.10-2)} \\
F_{px} &\le 0.4 S_{DS} I_e w_{px}  \quad \text{(ASCE Eq. 12.10-3)}
\end{align*}

These forces are shown in \autoref{tab:diaphragm_forces}

In [79]:
seismic_loads["cumulative floor weight"] = seismic_loads.iloc[::-1]["floor weight"].cumsum()
seismic_loads["F_x,cum"] = seismic_loads.iloc[::-1]["F_x"].cumsum()
seismic_loads["F_px lower bound"] = 0.2 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px upper bound"] = 0.4 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px initial"] = (
    seismic_loads["F_x,cum"] / seismic_loads["cumulative floor weight"] * seismic_loads["floor weight"]
)

# Assign F_px with limits from ASCE
seismic_loads["F_px"] = np.where(
    seismic_loads["F_px initial"] <= seismic_loads["F_px lower bound"],
    seismic_loads["F_px lower bound"],
    np.where(
        seismic_loads["F_px initial"] >= seismic_loads["F_px upper bound"],
        seismic_loads["F_px upper bound"],
        seismic_loads["F_px initial"],
    ),
)

In [80]:
column_name_map = {
    "floor weight": "$w_{px}$ [kip]",
    "cumulative floor weight": r"$\sum w_x$ [kip]",
    "F_x,cum": r"$\sum F_x$ [kip]",
    "F_px initial": "$F_{px,init}$ [kip]",
    "F_px lower bound": "$F_{px,lower}$ [kip]",
    "F_px upper bound": "$F_{px,upper}$ [kip]",
    "F_px": "$F_{px}$ [kip]",
}
display_table(
    dataframe=seismic_loads.iloc[:0:-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Forces",
    label="tab:diaphragm_forces",
)

,$w_{px}$ [kip],$\sum w_x$ [kip],$\sum F_x$ [kip],"$F_{px,init}$ [kip]","$F_{px,lower}$ [kip]","$F_{px,upper}$ [kip]",$F_{px}$ [kip]
Level,,,,,,,
Roof,421.490,421.490,72.387392,72.387392,57.086178,114.172357,72.387392
Level 4,613.642,1035.132,152.978136,90.687766,83.111051,166.222101,90.687766
Level 3,615.454,1650.586,207.692959,77.442473,83.356466,166.712932,83.356466
Level 2,619.078,2269.664,236.462302,64.497921,83.847297,167.694594,83.847297


With these diaphragm forces, we can design the diaphragm for the worst-case cantilever and simply-supported conditions.

The longest simply-supported span in the N-S direction is about 34.5 feet, while the longest cantilever span is about 32 feet. The depth of the diaphragm in the N-S direction is 67 feet. The longest simply-supported span in the E-W direction is about 5.5 feet, while the longest cantilever span is about 31 feet. The depth of the diaphragm in the E-W direction is 230 feet.

Diaphragm unit shear demands are shown in \autoref{tab:diaphragm_unit_shears}

In [81]:
# Set up diaphragm dataframe
levels = ["Roof", "Level 4", "Level 3", "Level 2"]
directions = ["N-S", "E-W"]
index = pd.MultiIndex.from_product(
    [levels, directions],
    names=["Level", "Direction"],
)
diaphragms = pd.DataFrame(index=index)
diaphragms_ns = diaphragms.xs("N-S", level="Direction").copy()
diaphragms_ew = diaphragms.xs("E-W", level="Direction").copy()

In [82]:
diaphragms_ns["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ns["area load"] = diaphragms_ns["F_px"] / float(A_level)
diaphragms_ns["L_simp"] = 34.5
diaphragms_ns["L_cant"] = 32
diaphragms_ns["depth"] = 67
diaphragms_ns["V_simp"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_simp"] / 2
diaphragms_ns["V_cant"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_cant"]
diaphragms_ns["v_simp"] = diaphragms_ns["V_simp"] / diaphragms_ns["depth"]
diaphragms_ns["v_cant"] = diaphragms_ns["V_cant"] / diaphragms_ns["depth"]

We will assume the diaphragm sheathing has the following properties:

- Unblocked
- WSP Sheathing
- 15/32" thick
- 10d common nails
- Plywood
- Nailed face is 2"
- Edge adjoining cases 2 through 6

The diaphragm capacity checks for this chosen sheathing are shown in \autoref{tab:diaphragm_capacities}

In [83]:
# Set baseline sheathing properties. We set the number of sheathed sides later
from structural_tools.seismic.timber.sheathing import SheathingApplication

diaphragm_sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
    minimum_nominal_width_of_nailed_face=2,
    adjoining_panel_edge_location=(2, 3, 4, 5, 6),
)

diaphragms_ns[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ns["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ns[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ns["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [84]:
diaphragms_ew["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ew["area load"] = diaphragms_ew["F_px"] / float(A_level)
diaphragms_ew["L_simp"] = 5.5
diaphragms_ew["L_cant"] = 31
diaphragms_ew["depth"] = float(L_floor)
diaphragms_ew["V_simp"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_simp"] / 2
diaphragms_ew["V_cant"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_cant"]
diaphragms_ew["v_simp"] = diaphragms_ew["V_simp"] / diaphragms_ew["depth"]
diaphragms_ew["v_cant"] = diaphragms_ew["V_cant"] / diaphragms_ew["depth"]


In [85]:
diaphragms_ew[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ew["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ew[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ew["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [86]:
idx = pd.IndexSlice
diaphragms = diaphragms.reindex(columns=diaphragms_ns.columns)
diaphragms_ns["Direction"] = "N-S"
diaphragms_ew["Direction"] = "E-W"
diaphragms_ns = diaphragms_ns.set_index("Direction", append=True)
diaphragms_ew = diaphragms_ew.set_index("Direction", append=True)
diaphragms.update(diaphragms_ns)
diaphragms.update(diaphragms_ew)

In [87]:
column_name_map = {
    "F_px": "$F_{px}$ [lbf]",
    "area load": "$w$ [psf]",
    "V_simp": "$V_{simp}$ [lbf]",
    "V_cant": "$V_{cant}$ [lbf]",
    "v_simp": "$v_{simp}$ [plf]",
    "v_cant": "$v_{cant}$ [plf]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Unit Shear Demands",
    label="tab:diaphragm_unit_shears",
)

$F_{px}$ [lbf]  $w$ [psf]  $V_{simp}$ [lbf]  \
Level   Direction                                                
Roof    N-S          72387.392027   4.697430       5429.054402   
        E-W          72387.392027   4.697430       2971.124300   
Level 4 N-S          90687.766441   5.884995       6801.582483   
        E-W          90687.766441   5.884995       3722.259070   
Level 3 N-S          83356.466100   5.409245       6251.734957   
        E-W          83356.466100   5.409245       3421.347489   
Level 2 N-S          83847.296988   5.441096       6288.547274   
        E-W          83847.296988   5.441096       3441.493533   

                   $V_{cant}$ [lbf]  $v_{simp}$ [plf]  $v_{cant}$ [plf]  
Level   Direction                                                        
Roof    N-S            10071.289326         81.030663        150.317751  
        E-W            33492.673923         12.917932        145.620321  
Level 4 N-S            12617.428374        101.516156        188.319826  
        E-W            41960.011338         16.183735        182.434832  
Level 3 N-S            11597.421370         93.309477        173.095841  
        E-W            38567.917151         14.875424        167.686596  
Level 2 N-S            11665.710885         93.858915        174.115088  
        E-W            38795.018009         14.963015        168.673991

In [88]:
column_name_map = {
    "v_simp": "$v_{simp}$ [plf]",
    "adjusted unit shear capacity simply supported": "$v_{cap,simp}$ [plf]",
    "shear dcr simply supported": "DCR$_{simp}$ [-]",
    "v_cant": "$v_{cant}$ [plf]",
    "adjusted unit shear capacity cantilever": "$v_{cap,cant}$ [plf]",
    "shear dcr cantilever": "DCR$_{cant}$ [-]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Capacity Checks",
    label="tab:diaphragm_capacities",
)

$v_{simp}$ [plf]  $v_{cap,simp}$ [plf]  DCR$_{simp}$ [-]  \
Level   Direction                                                             
Roof    N-S               81.030663            189.285714          0.428087   
        E-W               12.917932            189.285714          0.068246   
Level 4 N-S              101.516156            189.285714          0.536312   
        E-W               16.183735            189.285714          0.085499   
Level 3 N-S               93.309477            189.285714          0.492956   
        E-W               14.875424            189.285714          0.078587   
Level 2 N-S               93.858915            189.285714          0.495858   
        E-W               14.963015            189.285714          0.079050   

                   $v_{cant}$ [plf]  $v_{cap,cant}$ [plf]  DCR$_{cant}$ [-]  
Level   Direction                                                            
Roof    N-S              150.317751            189.285714          0.794132  
        E-W              145.620321            189.285714          0.769315  
Level 4 N-S              188.319826            189.285714          0.994897  
        E-W              182.434832            189.285714          0.963807  
Level 3 N-S              173.095841            189.285714          0.914469  
        E-W              167.686596            189.285714          0.885891  
Level 2 N-S              174.115088            189.285714          0.919853  
        E-W              168.673991            189.285714          0.891108

In [89]:
if (
    (diaphragms[["shear dcr simply supported", "shear dcr cantilever"]] <= 1).all().all()
):  # double .all() because 2 columns
    display_text(r"$\therefore$ All shear capacities are OK")
else:
    display_text("You must check for new diaphragm nailing.")

'$\\therefore$ All shear capacities are OK'